# Week 5 — Declustering and PyOcto Phase Association
Two steps: (1) decluster raw picks (remove near-duplicate same-phase picks
within a 5 s window, keeping the highest-probability one), (2) associate
declustered picks across stations into candidate events using PyOcto.

Velocity model: homogeneous (constant Vp/Vs), not the unavailable Mestel
(2023) 1D model — see progress log for the full justification. Values used:
Vp = 5.7 km/s (Stratford et al. 2002; Henrys et al. 2003, NIGHT refraction
survey, average TVZ crustal velocity) and Vp/Vs = 1.7 (Sherburn et al. 2003
and TVZ geothermal tomography studies), giving Vs = 5.7/1.7 = 3.35 km/s.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import obspy
import pyocto

PROJECT_DIR = os.path.expanduser("~/tvz_project")
PICKS_DIR = os.path.join(PROJECT_DIR, "picks", "eqtransformer")
INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")

picks_files = sorted(glob.glob(os.path.join(PICKS_DIR, "*_picks.csv")))
print(f"{len(picks_files)} station pick files found")

In [ ]:
# Load all picks, convert time to a UNIX timestamp (float) - this is the
# format PyOcto's associate() expects internally (confirmed from its own
# associate_seisbench() source: {"station": ..., "time": peak_time.timestamp,
# "probability": ..., "phase": ...})
all_picks = pd.concat([pd.read_csv(f) for f in picks_files], ignore_index=True)
all_picks["time_dt"] = pd.to_datetime(all_picks["time"])
all_picks["time_unix"] = all_picks["time_dt"].astype("int64") / 1e9  # ns -> s

print(f"Raw picks loaded: {len(all_picks)}")

In [ ]:
# Decluster: within each station+phase group, collapse picks that fall
# within 5 s of each other into one - keeping the highest-probability pick.
def decluster(df, window=5.0):
    kept = []
    for (station, phase), group in df.groupby(["station", "phase"]):
        group = group.sort_values("time_unix").reset_index(drop=True)
        cluster_start = 0
        for i in range(1, len(group) + 1):
            if i == len(group) or (group.loc[i, "time_unix"] - group.loc[i - 1, "time_unix"]) > window:
                cluster = group.loc[cluster_start:i - 1]
                kept.append(cluster.loc[cluster["probability"].idxmax()])
                cluster_start = i
    return pd.DataFrame(kept).reset_index(drop=True)

declustered = decluster(all_picks)
print(f"Before declustering: {len(all_picks)} picks")
print(f"After declustering:  {len(declustered)} picks "
      f"({100 * len(declustered) / len(all_picks):.1f}% retained)")

decl_out = os.path.join(PROJECT_DIR, "picks", "eqtransformer_declustered.csv")
declustered.to_csv(decl_out, index=False)
print(f"Saved: {decl_out}")

In [ ]:
# Homogeneous velocity model + associator, built from the region's lat/lon/depth
# limits directly (from_area handles the coordinate projection for us)
velocity_model = pyocto.associator.VelocityModel0D(
    p_velocity=5.7,
    s_velocity=5.7 / 1.7,
    tolerance=1.0,
)

associator = pyocto.associator.OctoAssociator.from_area(
    lat=(-40.0, -37.0),
    lon=(175.0, 178.0),
    zlim=(0, 40),           # km depth, matching the proposal's QuakeMigrate grid range
    velocity_model=velocity_model,
    time_before=300.0,      # proposal's stated 300 s time-slice overlap
    n_picks=10,             # proposal's stated minimum 10 picks per event
    n_p_picks=5,            # proposal's stated minimum 5 P picks
    n_s_picks=2,            # proposal's stated minimum 2 S picks
)
print("Associator configured")

# inventory_to_df is an instance method (confirmed by the previous TypeError,
# which showed it expects `self` then `inventory`) - call it on the
# associator instance itself, not the class.
inventory = obspy.read_inventory(INVENTORY_FILE)
stations = associator.inventory_to_df(inventory)
print(f"{len(stations)} stations converted to PyOcto format")
print(stations.head())


In [ ]:
# Build the pick dataframe in the exact format associate() expects.
# Fixed: select the unique "time_unix" column BEFORE renaming, not after -
# declustered already has an original "time" column (the datetime string),
# so renaming time_unix -> time on the whole dataframe first created two
# columns both named "time", which is why row["time"] returned two values.
pick_df = declustered[["station", "time_unix", "probability", "phase"]].rename(
    columns={"time_unix": "time"}
)

events, assignments = associator.associate(pick_df, stations)

print(f"Candidate events found: {len(events)}")
print(f"Picks assigned to events: {len(assignments)} of {len(pick_df)} declustered picks")
print(events.head())


In [ ]:
events_out = os.path.join(PROJECT_DIR, "catalogs", "eqtransformer_pyocto_events.csv")
assignments_out = os.path.join(PROJECT_DIR, "catalogs", "eqtransformer_pyocto_assignments.csv")
os.makedirs(os.path.dirname(events_out), exist_ok=True)

events.to_csv(events_out, index=False)
assignments.to_csv(assignments_out, index=False)
print(f"Saved: {events_out}")
print(f"Saved: {assignments_out}")